<div style="font-size:2.3em;font-weight:600;margin-bottom:10px">Agentic Artificial Intelligence</div>
<div style="font-size:1.7em;margin-bottom:14px">Unit 05 · Integrating tools</div>

Today our agent gets **tools**.

<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px 32px"><div><b>1</b>&nbsp; Opening: a model in a closed room</div><div><b>5</b>&nbsp; The agent does it for us</div><div><b>2</b>&nbsp; Building a tool with @tool</div><div><b>6</b>&nbsp; Your exercise: your own tool</div><div><b>3</b>&nbsp; Giving the tool to the model</div><div><b>7</b>&nbsp; Wrap-up</div><div><b>4</b>&nbsp; Tools as classes, and why descriptions matter</div><div></div></div>

<div style="height:44px"></div>

## Part 1 of 7 · Opening: a model in a closed room

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**Why does a model need tools?**

#### 1.1 · Imagine this

Think of the model as a very smart person in a **closed room**.

No phone, no computer, no window. Only a small gap under the door.

**Text comes in, and text goes out.**

Outside the room there are **tools**, for example a calculator.

#### 1.2 · Four limits of a language model

**1.** No fresh data: training ended at some point

**2.** No reliable math

**3.** No actions in the outside world

**4.** **Hallucination:** the model invents something and sounds completely sure

#### 1.3 · Today's plan

**1.** We build a tool

**2.** We give it to the model, and we watch what it does

**3.** We let an agent do the work for us

**4.** Your exercise

<div style="height:44px"></div>

## Part 2 of 7 · Building a tool with @tool

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What makes a function a tool?**

#### 2.1 · A tool is a function with a label

A normal Python function, with a label that the model can read.

The easy way to make one is the **decorator** `@tool`: a line with an at sign above the function.

In [ ]:
# 2.2 · The decorator
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together.

    Use this tool when you need to multiply two integers.

    Args:
        a: The first number
        b: The second number

    Returns:
        The product of a and b
    """
    return a * b

In [ ]:
# 2.3 · What the decorator made
print("Tool created with decorator:")
print("=" * 50)
print(f"Name: {multiply.name}")
print(f"Description: {multiply.description}")
print(f"\nTesting the tool:")
print(f"multiply(5, 3) = {multiply.invoke({'a': 5, 'b': 3})}")

In [ ]:
# 2.4 · A name that we choose
@tool("power_calculator")
def power(base: float, exponent: float) -> float:
    """Calculate base raised to the power of exponent.

    Use this tool for exponentiation operations.

    Args:
        base: The base number
        exponent: The exponent (power)

    Returns:
        base raised to the power of exponent
    """
    return base ** exponent

In [ ]:
# 2.5 · What this decorator made
print("Custom named tool:")
print("=" * 50)
print(f"Name: {power.name}")
print(f"Description: {power.description[:80]}...")
print(f"\nTesting: power(2, 8) = {power.invoke({'base': 2, 'exponent': 8})}")

<div style="height:44px"></div>

## Part 3 of 7 · Giving the tool to the model

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What does the model do with a tool?**

In [ ]:
# 3.1 · The key and the model
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

In [ ]:
# 3.2 · The menu, and a question
llm_with_tools = llm.bind_tools([power, multiply])

query = HumanMessage(
    content="What is 2 to the power of 8? Provide the result in a sentence."
)
ai_msg = llm_with_tools.invoke([query])

print(ai_msg.tool_calls)

#### 3.3 · One thing to remember

<details>
<summary>Click to show</summary>

**The model never runs a tool. It only asks.**

**Our code runs the tool and gives back the result.**

</details>

In [ ]:
# 3.4 · The next step
tool_message = ToolMessage(
    content=power.invoke(ai_msg.tool_calls[0]["args"]),
    tool_call_id=ai_msg.tool_calls[0]["id"],
)

In [ ]:
# 3.5 · The last step
full_conversation = [query, ai_msg, tool_message]

final_response = llm_with_tools.invoke(full_conversation)
print(final_response.text)

<div style="height:44px"></div>

## Part 4 of 7 · Tools as classes, and why descriptions matter

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**Does the model care how we build a tool?**

In [ ]:
# 4.1 · A tool as a class
from langchain_core.tools import BaseTool

class PowerTool(BaseTool):
    name: str = "power_calculator"
    description: str = """Calculate base raised to the power of exponent.
    
    Use this tool for exponentiation operations.
    
    Args:
        base: The base number
        exponent: The exponent (power)
        
    Returns:
        base raised to the power of exponent
    """

    def _run(self, base: float, exponent: float) -> float:
        return base ** exponent

In [ ]:
# 4.2 · The same question
tools = [PowerTool()]

llm_with_tools = llm.bind_tools(tools)

query = HumanMessage(content="What is 2 to the power of 8?")
ai_msg = llm_with_tools.invoke([query])

print(ai_msg.tool_calls)

In [ ]:
# 4.3 · The calculator in our package
from agentic_ai.tools import CalculatorTool

calc = CalculatorTool()

print("Tool Information:")
print(f"Name: {calc.name}")
print(f"Description: {calc.description}")
print(f"\nTool String Representation: {calc}")

#### 4.4 · Inside the calculator

From our package, `agentic_ai/tools/calculator.py`:

```python
def _run(self, expression: str) -> Union[float, int, str]:
    try:
        node = ast.parse(expression, mode='eval')
        return self._eval_node(node.body)
    except Exception as e:
        return f"Error: {str(e)}"
```

In [ ]:
# 4.5 · The same question, with the calculator
llm_with_tools = llm.bind_tools([CalculatorTool()])

query = HumanMessage(content="What is 2 to the power of 8?")
ai_msg = llm_with_tools.invoke([query])

print(ai_msg.tool_calls)

In [ ]:
# 4.6 · Run the calculator, and answer
tool_message = ToolMessage(
    content=CalculatorTool().run(ai_msg.tool_calls[0]["args"]),
    tool_call_id=ai_msg.tool_calls[0]["id"],
)

final_response = llm_with_tools.invoke([query, ai_msg, tool_message])
print(final_response.text)

#### 4.7 · Decorator or class?

**A simple function:** take the decorator.

**You need settings:** take a class.

---

A tool has three parts that the model reads: **1.** the name, **2.** the description, **3.** the inputs.

The description is **very important**: the model reads it to decide.

<div style="height:44px"></div>

## Part 5 of 7 · The agent does it for us

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What if the model needs two tools, one after the other?**

In [ ]:
# 5.1 · The agent we had before
from agentic_ai.agents.simple_agent import SimpleAgent

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

SimpleAgent(llm, 'simple_agent').graph

#### 5.2 · The idea behind our new agent

The class `ToolAgent` follows the idea **Thought, Action, Observation**. The short name is **ReAct**.

#### 5.3 · What must we add to the picture?

<details>
<summary>Click to show</summary>

```
┌─────────┐
│  START  │
└────┬────┘
     │
     ▼
┌─────────┐
│  agent  │
└────┬────┘
     │
     ├─── Conditional Edge ───┐
     │                        │
     ▼                        ▼
┌─────────┐              ┌─────────┐
│  tools  │              │   END   │
└────┬────┘              └─────────┘
     │
     │ (after tool execution)
     │
     └───► back to agent ───┐
                            │
                            ▼
                       (loop until done)
```

</details>

In [ ]:
# 5.4 · The parts for our new agent
from agentic_ai.agents.tool_agent import ToolAgent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")
memory = InMemorySaver()

calculator_tool = CalculatorTool()

langchain_tools = [
    calculator_tool#.to_langchain_tool()
]

In [ ]:
# 5.5 · Build the agent
agent = ToolAgent(
    llm=llm,
    tools=langchain_tools,
    name="MathBot",
    checkpointer=memory
)

print("Agent created with tools:")
print("=" * 50)
for tool_name in agent.get_available_tools():
    print(f"  - {tool_name}")

In [ ]:
# 5.6 · Draw its graph
print("ToolAgent Graph Structure:")
print("=" * 50)
agent.graph

#### 5.7 · Inside the class: the start

From our package, `agentic_ai/agents/tool_agent.py`. The class is `ToolAgent(BaseAgent)`.

```python
def __init__(
    self,
    llm: BaseChatModel,
    tools: List[BaseTool],
    name: str = "ToolAgent",
    system_prompt: Optional[str] = None,
    checkpointer: Optional[BaseCheckpointSaver] = None
):
    self.tools = tools
    self.tool_node = ToolNode(tools)

    default_prompt = (
        "You are a helpful AI assistant with access to various tools. "
        "Use the available tools when they can help answer the user's question or complete their request. "
        "Always explain what you're doing and provide clear, helpful responses."
    )

    super().__init__(
        llm=llm.bind_tools(tools),  # Bind tools to the LLM
        name=name,
        system_prompt=system_prompt or default_prompt,
        checkpointer=checkpointer
    )
```

#### 5.8 · Inside the class: the graph

```python
def _build_graph(self) -> None:
    workflow = StateGraph(State)

    workflow.add_node("agent", self.call_model)
    workflow.add_node("tools", self.tool_node)

    workflow.set_entry_point("agent")

    workflow.add_conditional_edges(
        "agent",
        self.should_continue,
        {
            "continue": "tools",
            "end": END
        }
    )

    workflow.add_edge("tools", "agent")

    self.compile_graph(workflow)
```

#### 5.9 · Inside the class: the decision

```python
def should_continue(self, state: State) -> str:
    last_message = state["messages"][-1]

    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "continue"
    else:
        return "end"
```

In [ ]:
# 5.10 · A real test
print("\n\nTest: Complex calculation")
print("=" * 50)
response = agent.run("Calculate (10 + 5) * 3 - 7", thread_id="test")
print(response['messages'][-1].text)

#### 5.11 · Did the agent use the tool?

The answer alone does not tell us.

In [ ]:
# 5.12 · Look through the messages
from langchain_core.messages import AIMessage

for msg in response['messages']:
    if isinstance(msg, AIMessage) and msg.tool_calls:
        print('The agent used the tool with the following message:')
        print(msg)

In [ ]:
# 5.13 · A full conversation
response = agent.run("What is 15 * 7?", thread_id="trace_example")

print("Full Conversation History:")
print("=" * 50)
history = agent.get_conversation_history(thread_id="trace_example")

for i, msg in enumerate(history):
    print(f"\n{i+1}. {msg.__class__.__name__}:")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"   Tool Calls: {msg.tool_calls}")
    if hasattr(msg, 'tool_call_id'):
        print(f"   Tool Call ID: {msg.tool_call_id}")
    print(f"   Content: {msg.text[:200]}..." if len(msg.text) > 200
          else f"   Content: {msg.text}")

<div style="height:44px"></div>

## Part 6 of 7 · Your exercise: your own tool

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What do you build at home?**

#### 6.1 · Task 1: a tool called `text_length`

It takes a text, and it returns the number of **characters**, **words** and **sentences**.

It inherits from **our** `BaseTool`, from `agentic_ai.tools`. That is not the LangChain class with the same name.

It needs an `__init__` with a name and a description, and two methods: `run` and `to_langchain_tool`.

Also handle an empty text and `None`, and give the tool a clear description.

#### 6.2 · An example, and your starting point

From the calculator in our package, `agentic_ai/tools/calculator.py`:

```python
def to_langchain_tool(self) -> Tool:
    return Tool(
        name=self.name,
        description=self.description,
        func=self._run
    )
```

The calculator has `_run`. Your tool has `run`, so you give it `self.run`.

```python
# TODO: Create your TextLengthTool class here
from agentic_ai.tools import BaseTool

# Test your tool
# tool = TextLengthTool()
# print(tool.run("Hello, World!"))
# print(tool.run(""))
# print(tool.run("This is a longer text string."))
```

#### 6.3 · Task 2: your tool and the calculator in one `ToolAgent`

**Your tool must be converted** with `to_langchain_tool`.

Test the agent with these five:

- What is the length of the text 'Hello, World!'?
- How many words are in 'This is a test sentence'?
- Calculate 15 * 7
- What is the length and word count of 'Agentic AI is fascinating'?
- Write a poem about agentic AI that is exactly 7 sentences long

#### 6.4 · Task 3, and the bonus tasks

Print the full history of one of your questions.

How many messages are there? Which ones are notes (`tool_calls`)? Which ones are results (`tool_call_id`)? What is the final answer?

**Task 4 (bonus):** a `TextAnalysisTool`, with more statistics.

**Task 6 (bonus):** your own `CustomToolAgent`, with a limit on tool calls.

All details are in your own notebook.

<div style="height:44px"></div>

## Part 7 of 7 · Wrap-up

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span></div>

**What should you remember from today?**

#### 7.1 · Five things

**1.** A tool has a name, a description and inputs. The model reads those three, and it never sees the code.

**2.** `bind_tools` gives the model the menu.

**3.** The model never runs a tool. It only asks. Our code runs the tool and gives back the result.

**4.** An agent does this in a loop. The model writes a note or no note, and a conditional edge then sends us to the tools or to the end.

**5.** A description is a prompt, so write it with care.

---

**At home:** Tasks 1, 2 and 3.

**Next time, Unit 6:** tools that run in other programs, with **MCP**.